# 1. Setup

### 1.1 Install deps & packages

In [1]:
%pip install numpy pandas matplotlib kagglehub
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import kagglehub
import shutil
import os

You should consider upgrading via the '/Users/Hita/Desktop/Transaction-fraud-detection/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


/Users/Hita/Desktop/Transaction-fraud-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Download dataset

In [2]:
try:
    os.mkdir('original_dataset')
    kagglehub.dataset_download("zahranusratt/banking-fraud-detection-dataset", output_dir='original_dataset')
except FileExistsError:
    pass

# Remove uesless metadata
shutil.rmtree('original_dataset/.complete/', ignore_errors=True)

print('Dataset downloaded')

Dataset downloaded


# 2. Data pipeline

### 2.1 Load csv into dataframe

In [3]:
fraud_data = pd.read_csv('original_dataset/bank_fraud.csv')

print(f"Shape\n{fraud_data.shape}\n")
print(f"Head\n{fraud_data.head()}")
print("\nColumns")
for column_number, column_name in enumerate(fraud_data.columns, start=1):
    print(column_number, column_name)

Shape
(1000000, 26)

Head
  transaction_id   customer_id transaction_date transaction_time  hour_of_day  \
0  TXN0000000001  CUST00121959       2023-08-17         21:13:00           21   
1  TXN0000000002  CUST00146868       2024-02-06         05:16:00            5   
2  TXN0000000003  CUST00131933       2024-06-28         12:15:00           12   
3  TXN0000000004  CUST00103695       2023-03-16         02:53:00            2   
4  TXN0000000005  CUST00119880       2024-07-12         12:39:00           12   

   is_weekend  is_night_transaction country       city merchant_category  ...  \
0           0                     0     USA     London           Grocery  ...   
1           0                     1      UK   New York        Healthcare  ...   
2           0                     0  Canada      Delhi           Grocery  ...   
3           0                     1  France      Tokyo         Utilities  ...   
4           0                     0  Canada  Melbourne          Clothing  ...   



In [4]:
fraud_data_original = fraud_data.copy(deep=True)

print("Records:", fraud_data_original.shape[0])
print("Columns:", fraud_data_original.shape[1])

print("\nData-type counts:")
print(fraud_data_original.dtypes.value_counts())

Records: 1000000
Columns: 26

Data-type counts:
int64      11
object     10
float64     5
Name: count, dtype: int64


In [5]:
summary = pd.DataFrame({
    'dtype': fraud_data_original.dtypes,
    'sample_values': [fraud_data_original[col].dropna().unique()[:3] for col in fraud_data_original.columns]
})
print(summary.to_string())

                            dtype                                         sample_values
transaction_id             object         [TXN0000000001, TXN0000000002, TXN0000000003]
customer_id                object            [CUST00121959, CUST00146868, CUST00131933]
transaction_date           object                  [2023-08-17, 2024-02-06, 2024-06-28]
transaction_time           object                        [21:13:00, 05:16:00, 12:15:00]
hour_of_day                 int64                                           [21, 5, 12]
is_weekend                  int64                                                [0, 1]
is_night_transaction        int64                                                [0, 1]
country                    object                                     [USA, UK, Canada]
city                       object                             [London, New York, Delhi]
merchant_category          object                      [Grocery, Healthcare, Utilities]
payment_method             objec

2.  Numeric ranges, missing values, duplicates, and inconsistent entries

In [6]:


print("\n" + "="*60)
print("NUMERIC RANGES")
print("="*60)
print(fraud_data_original.describe())

print("\n" + "="*60)
print("CATEGORICAL SUMMARY")
print("="*60)
print(fraud_data_original.describe(include='object'))

print("\n" + "="*60)
print("UNIQUE VALUES (categorical columns)")
print("="*60)
for col in fraud_data_original.select_dtypes(include="object").columns:
    print(f"\n{col}: {fraud_data_original[col].nunique()} unique values")
    print(fraud_data_original[col].unique()[:15])

print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
missing = fraud_data_original.isna().sum()
missing_pct = (missing / len(fraud_data_original) * 100).round(2)
missing_summary = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(missing_summary[missing_summary["missing"] > 0].sort_values("missing", ascending=False))

print("\n" + "="*60)
print("DUPLICATE ROWS")
print("="*60)
print("Full duplicate rows:", fraud_data_original.duplicated().sum())
if "transaction_id" in fraud_data_original.columns:
    print("Duplicate transaction_id:", fraud_data_original["transaction_id"].duplicated().sum())


NUMERIC RANGES
          hour_of_day      is_weekend  is_night_transaction    customer_age  \
count  1000000.000000  1000000.000000        1000000.000000  1000000.000000   
mean        11.496978        0.286022              0.375057       41.771678   
std          6.923751        0.451900              0.484138       13.424588   
min          0.000000        0.000000              0.000000       18.000000   
25%          5.000000        0.000000              0.000000       32.000000   
50%         11.000000        0.000000              0.000000       42.000000   
75%         18.000000        1.000000              1.000000       51.000000   
max         23.000000        1.000000              1.000000       85.000000   

         credit_score  account_age_years  account_balance  transaction_amount  \
count  1000000.000000     1000000.000000    1000000.00000      1000000.000000   
mean       679.028781           4.987911      16594.25442          204.724665   
std         78.828748        

# 3. Data Cleansing and Transformation

### 3.1 Dropping columns

Looking at the dataset, we can see the transaction ID is unique to every row, therefore it won't be useful in deriving any value.
We can use the customer contry to identify if the transaction took place in a high risk country, for this reason we can drop the city as it's too spesific.

In [7]:
fraud_data = fraud_data.drop(columns=['transaction_id', 'country', 'city'])

fraud_data.head()

,customer_id,transaction_date,transaction_time,hour_of_day,is_weekend,is_night_transaction,merchant_category,payment_method,device_type,customer_age,...,transaction_amount,num_prev_transactions,transaction_freq_monthly,distance_from_home_km,time_since_last_txn_hrs,is_international,failed_attempts,pin_changed_recently,is_fraud,fraud_type
0,CUST00121959,2023-08-17,21:13:00,21,0,0,Grocery,Bank Transfer,POS Terminal,18,...,39.49,157,23,52.7,10.20,0,0,0,0,NaN
1,CUST00146868,2024-02-06,05:16:00,5,0,1,Healthcare,Cheque,Desktop,30,...,153.71,153,23,0.9,12.47,0,0,0,0,NaN
2,CUST00131933,2024-06-28,12:15:00,12,0,0,Grocery,Crypto,Mobile,20,...,118.20,161,20,9.2,0.08,0,1,0,0,NaN
3,CUST00103695,2023-03-16,02:53:00,2,0,1,Utilities,Debit Card,Mobile,29,...,49.50,160,25,14.8,17.94,1,0,1,1,Synthetic Identity
4,CUST00119880,2024-07-12,12:39:00,12,0,0,Clothing,Debit Card,Desktop,49,...,30.74,134,18,38.9,2.16,0,0,0,0,NaN


### 3.2 Normalising data

Since a lot of these columns are categories we can convert them to a number using a dict

In [8]:
# A list of categorical columns
categorical_columns = ['merchant_category', 'payment_method', 'device_type', 'fraud_type']

# Create a dict of the enum values for each category
category_values = {}

for category in categorical_columns:
    category_values[category] = fraud_data[category].unique().tolist()

# Apply the normalisation
for i in range(fraud_data.shape[0]):
    for key in category_values:
        fraud_data.at[i, key] = category_values[key].index(fraud_data.loc[i][key])
    
fraud_data.head()

,customer_id,transaction_date,transaction_time,hour_of_day,is_weekend,is_night_transaction,merchant_category,payment_method,device_type,customer_age,...,transaction_amount,num_prev_transactions,transaction_freq_monthly,distance_from_home_km,time_since_last_txn_hrs,is_international,failed_attempts,pin_changed_recently,is_fraud,fraud_type
0,CUST00121959,2023-08-17,21:13:00,21,0,0,0,0,0,18,...,39.49,157,23,52.7,10.20,0,0,0,0,0
1,CUST00146868,2024-02-06,05:16:00,5,0,1,1,1,1,30,...,153.71,153,23,0.9,12.47,0,0,0,0,0
2,CUST00131933,2024-06-28,12:15:00,12,0,0,0,2,2,20,...,118.20,161,20,9.2,0.08,0,1,0,0,0
3,CUST00103695,2023-03-16,02:53:00,2,0,1,2,3,2,29,...,49.50,160,25,14.8,17.94,1,0,1,1,1
4,CUST00119880,2024-07-12,12:39:00,12,0,0,3,3,1,49,...,30.74,134,18,38.9,2.16,0,0,0,0,0


### 3.3 Creating a new feature for identifying high risk transactions

We can create a new feature called `high_risk` for what is roughly a high risk transaction, this would be defined by
- `account_age_years` <= 1
- `time_since_last_txn_hrs` <= 1
- `is_international` == 1
- `pin_changed_recently` == 1
- `transaction_amount` >= 100

In [9]:
fraud_data['high_risk'] = np.where(
    (fraud_data['account_age_years'] <= 1) &
    (fraud_data['time_since_last_txn_hrs'] <= 1) &
    (fraud_data['is_international'] == 1) &
    (fraud_data['transaction_amount'] >= 100),
    1, 0)

filtered_df = fraud_data[fraud_data['high_risk'] == 1]

print(f"Found {filtered_df.shape[0]} high risk transactions")

fraud_data['high_risk'].describe()

Found 908 high risk transactions


count    1000000.000000
mean           0.000908
std            0.030119
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            1.000000
Name: high_risk, dtype: float64

## 4. Exploratory Data Analysis

This section explores the cleaned bank transaction dataset using descriptive statistics and visualisations. The purpose is to identify important distributions, relationships, unusual patterns and possible class imbalance that could support fraud detection.

### 4.1 Basic Dataset Overview

Before beginning the exploratory analysis, the cleaned dataset is checked to confirm its size, columns and data types.

In [10]:
# Check the structure of the cleaned dataset

print("Cleaned dataset shape:", fraud_data.shape)
print("Number of records:", fraud_data.shape[0])
print("Number of columns:", fraud_data.shape[1])

print("\nData-type counts:")
print(fraud_data.dtypes.value_counts())

fraud_data.info()

Cleaned dataset shape: (1000000, 24)
Number of records: 1000000
Number of columns: 24

Data-type counts:
int64      12
object      7
float64     5
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 24 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   customer_id               1000000 non-null  object 
 1   transaction_date          1000000 non-null  object 
 2   transaction_time          1000000 non-null  object 
 3   hour_of_day               1000000 non-null  int64  
 4   is_weekend                1000000 non-null  int64  
 5   is_night_transaction      1000000 non-null  int64  
 6   merchant_category         1000000 non-null  object 
 7   payment_method            1000000 non-null  object 
 8   device_type               1000000 non-null  object 
 9   customer_age              1000000 non-null  int64  
 10  credit_score              

The cleaned dataset contains 1,000,000 records and 24 columns. It includes 12 integer columns, 5 decimal columns and 7 object columns. The non-null counts show that all 24 columns contain 1,000,000 values, meaning the cleaned dataset is ready for further exploratory analysis.